# Correlation

In [ ]:
library(tidyverse)
options(repr.matrix.max.rows = 10) # so that we get shorter print outs of our tables in the Jupyter notebook
set.seed(42) # random seed so your random plots look just like my random plots

## Approach to Prediction

Based on incomplete information, one way to predict an outcome for an individual is to find others who are like that individual and whose outcomes you know, then use those outcomes as the basis of your prediction.

In [ ]:
families <- read_csv("family_heights.csv")
families

In [ ]:
heights <- families |>
  mutate(parentAvg = (father + mother) / 2) |>
  select(parentAvg, childHeight)
heights

In [ ]:
ggplot(heights, aes(x = parentAvg, y = childHeight)) +
  geom_point(alpha = 0.5) +
  labs(x = "Parent average height", y = "Child height") +
  theme_minimal(base_size=16)

In [ ]:
# Look at children whose parent average is near 68
nearby <- heights |>
  filter(parentAvg >= 67.5, parentAvg < 68.5)

nearbyMean <- mean(nearby$childHeight)
nearbyMean

In [ ]:
ggplot(heights, aes(x = parentAvg, y = childHeight)) +
  geom_point(alpha = 0.5) +
  geom_vline(xintercept = c(67.5, 68.5), color = "red", linewidth = 0.8) +
  annotate("point", x = 68, y = nearby_mean, color = "red", size = 4) +
  labs(x = "Parent average height", y = "Child height") +
  theme_minimal(base_size=16)

In [ ]:
predictChildHeight <- function(h) {
  nearby <- heights |>
    filter(parentAvg >= h - 0.5, parentAvg < h + 0.5)
  mean(nearby$childHeight)
}

heights_with_pred <- heights |>
  mutate(prediction = map_dbl(parentAvg, predictChildHeight))

# This map_dbl() is applying the function predictChildHeight() to the parentAvg variable.
# For each row, find children with a similar parent average and take the mean
# map_dbl() applies the function to each value of parentAvg and returns a number

In [ ]:
ggplot(heights_with_pred, aes(x = parentAvg)) +
  geom_point(aes(y = childHeight), alpha = 0.3) +
  geom_point(aes(y = prediction), color = "gold", size = 1) +
  labs(x = "Parent average height", y = "Height") +
  theme_minimal(base_size=16)

## Association

In [ ]:
hybrid <- read_csv("hybrid.csv")
hybrid

In [ ]:
hybrid |> arrange(desc(msrp))

In [ ]:
ggplot(hybrid, aes(x = mpg, y = msrp)) +
  geom_point() +
  theme_minimal(base_size=16)

In [ ]:
ggplot(hybrid, aes(x = acceleration, y = msrp)) +
  geom_point() +
  theme_minimal(base_size=16)

In [ ]:
suv <- hybrid |>
  filter(class == "SUV")
nrow(suv)

In [ ]:
ggplot(suv, aes(x = acceleration, y = msrp)) +
  geom_point() +
  theme_minimal(base_size=16)

In [ ]:
ggplot(suv, aes(x = mpg, y = msrp)) +
  geom_point() +
  theme_minimal(base_size=16)

### Standard units

Plotting in standard units lets us compare the degree of linearity across different scatter diagrams.

In [ ]:
suv |>
  mutate(
    mpg_su = (mpg - mean(mpg)) / sd(mpg),
    msrp_su = (msrp - mean(msrp)) / sd(msrp)
  ) |>
  ggplot(aes(x = mpg_su, y = msrp_su)) +
  geom_point() +
  coord_cartesian(xlim = c(-3, 3), ylim = c(-3, 3)) +
  labs(x = "mpg (standard units)", y = "msrp (standard units)") +
  theme_minimal(base_size=16)

In [ ]:
suv |>
  mutate(
    accel_su = (acceleration - mean(acceleration)) / sd(acceleration),
    msrp_su = (msrp - mean(msrp)) / sd(msrp)
  ) |>
  ggplot(aes(x = accel_su, y = msrp_su)) +
  geom_point() +
  coord_cartesian(xlim = c(-3, 3), ylim = c(-3, 3)) +
  labs(x = "acceleration (standard units)", y = "msrp (standard units)") +
  theme_minimal(base_size=16)

## Correlation

The correlation coefficient $r$ measures linear association:

- $-1 \leq r \leq 1$
- $r = 1$: perfect positive line
- $r = -1$: perfect negative line
- $r = 0$: no linear association (uncorrelated)

In [ ]:
r_scatter <- function(r) {
  n <- 1000
  x <- rnorm(n)
  z <- rnorm(n)
  y <- r * x + sqrt(1 - r^2) * z
  tibble(x = x, y = y) |>
    ggplot(aes(x, y)) +
    geom_point(alpha = 0.4, size = 1) +
    coord_cartesian(xlim = c(-4, 4), ylim = c(-4, 4)) +
    labs(title = paste("r ≈", r)) +
    theme_minimal(base_size=16)
}

In [ ]:
r_scatter(0.9)

In [ ]:
r_scatter(0.25)

In [ ]:
r_scatter(0)

In [ ]:
r_scatter(-0.55)

### Calculating $r$ step by step

$r$ is the average of the products of the two variables, when both variables are measured in standard units.

In [ ]:
x <- 1:6
y <- c(2, 3, 1, 5, 2, 7)
t <- tibble(x = x, y = y)
t

In [ ]:
ggplot(t, aes(x, y)) +
  geom_point(color = "red", size = 3) +
  theme_minimal(base_size=16)

**Step 1.** Convert each variable to standard units.

In [ ]:
t_su <- t |>
  mutate(
    x_su = (x - mean(x)) / sd(x),
    y_su = (y - mean(y)) / sd(y)
  )
t_su

In [ ]:
ggplot(t_su, aes(x = x_su, y = y_su)) +
  geom_point(color = "red", size = 3) +
  labs(x = "x (standard units)", y = "y (standard units)") +
  theme_minimal(base_size=16)

**Step 2.** Multiply each pair of standard units.

In [ ]:
t_product <- t_su |>
  mutate(product_su = x_su * y_su)
t_product

**Step 3.** $r$ is the average of the products.

In [ ]:
r <- mean(t_product$product_su)
r

### Using R's built-in `cor()`

R provides `cor()` which computes the Pearson correlation. It uses the sample standard deviation ($n-1$) rather than the population standard deviation ($n$). This is a correction we will explore in a future assignment where you program your own correlation function.

In [ ]:
cor(t$x, t$y)

In [ ]:
# Correlation of SUV mpg vs msrp
cor(suv$mpg, suv$msrp)

In [ ]:
# Correlation of SUV acceleration vs msrp
cor(suv$acceleration, suv$msrp)

### Switching axes

$r$ is unaffected by switching which variable is on which axis.

In [ ]:
cor(t$x, t$y)

In [ ]:
cor(t$y, t$x)

### Nonlinearity

Correlation only measures *linear* association. A perfect quadratic relationship can have $r = 0$:

In [ ]:
nonlinear <- tibble(
  x = seq(-4, 4, by = 0.5),
  y = x^2
)

ggplot(nonlinear, aes(x, y)) +
  geom_point(color = "red", size = 3) +
  theme_minimal(base_size=16)

In [ ]:
cor(nonlinear$x, nonlinear$y)

### Outliers

Outliers can have a big effect on correlation.

In [ ]:
line_data <- tibble(x = 1:4, y = 1:4)

ggplot(line_data, aes(x, y)) +
  geom_point(color = "red", size = 3) +
  theme_minimal(base_size=16)

In [ ]:
cor(line_data$x, line_data$y)

In [ ]:
outlier_data <- tibble(x = 1:5, y = c(1, 2, 3, 4, 0))

ggplot(outlier_data, aes(x, y)) +
  geom_point(color = "red", size = 3) +
  theme_minimal(base_size=16)

In [ ]:
cor(outlier_data$x, outlier_data$y)

### Ecological correlations

Correlations based on group averages can be misleadingly high. The correlation below is between *state average* SAT scores — not individual students.

In [ ]:
sat2014 <- read_csv("sat2014.csv") |>
  arrange(State)
sat2014

In [ ]:
ggplot(sat2014, aes(x = `Critical Reading`, y = Math)) +
  geom_point() +
  labs(title = "SAT Scores by State (2014)") +
  theme_minimal(base_size=16)

In [ ]:
cor(sat2014$`Critical Reading`, sat2014$Math)